# 南山一号：ZCU104 三种运算与原工程回归

同目录需要：nanshan_full.bit、nanshan_full.hwh、board_config.json、nanshan.py、check_environment.py、original_cases.zip 和本 Notebook。

按顺序执行。硬件时钟以配套配置为准；不要混用此前 64×64 乘法版本的文件。逻辑矩阵上限 512，通过 PYNQ 调度 64×64 输出分块，K 维在 FPGA 内完整累加。本版没有 DMA/PL 直接访问 DDR 的数据通路。


In [2]:
%run check_environment.py

{
  "machine": "aarch64",
  "system": "Linux",
  "python": "3.12.3",
  "board_model": "ZynqMP ZCU104 RevA",
  "pynq_version": "4.0.0",
  "overlay_mmio_importable": true,
  "clocks_api_importable": true,
  "python_environment_ready": true
}
PYENV_READY (hardware compatibility still requires testing)


In [3]:
from nanshan import extract_original_cases
extract_original_cases()

ORIGINAL_CASES_READY: 28 checksummed numerical fixtures


## 先验证小矩阵乘法、加法、减法
每项必须 passed=True、mismatches=0；成功后再继续。

In [4]:
import numpy as np
from nanshan import Accelerator
acc = Accelerator()
A = np.array([[1., -2., .5], [0., 1., 0.]])
B_mul = np.array([[2., 0.], [1., -1.], [0., 4.]])
B_add = np.array([[0., 1., .5], [2., 1., 0.]])
for mode, B in [('spmm', B_mul), ('add', B_add), ('sub', B_add)]:
    C, report = acc.calculate(A, B, mode=mode)
    print(mode, C, {key: report[key] for key in ['passed','mismatches','tiles','pl_time_ms']}, sep='\n')
    assert report['passed'], 'Stop and preserve this output'
print('THREE_MODE_SMOKE_PASS')


spmm
[[ 0.  4.]
 [ 1. -1.]]
{'passed': True, 'mismatches': 0, 'tiles': 1, 'pl_time_ms': 0.08354083540835408}


add
[[ 1. -1.  1.]
 [ 2.  2.  0.]]
{'passed': True, 'mismatches': 0, 'tiles': 1, 'pl_time_ms': 0.08252082520825209}


sub
[[ 1. -3.  0.]
 [-2.  0.  0.]]
{'passed': True, 'mismatches': 0, 'tiles': 1, 'pl_time_ms': 0.08252082520825209}
THREE_MODE_SMOKE_PASS


## 原工程 28 个用例

直接与原工程 golden_c.hex 转存的位模式核对，覆盖三种运算、空行、B-only、抵消、行重排数据、非整组行数和 K 多块。会生成 full_results 文件夹；测试期间等待代码执行完成，不要重复点击运行。


In [5]:
from nanshan import run_original_suite
original_reports = run_original_suite()

PL clock register readback: 49.999 MHz


add_a_empty_rows128 {"passed": true, "mode": "add", "shape": [128, 128, 128], "tiles": 4, "mismatches": 0, "pl_time_ms": 0.3844838448384484}


add_b_empty_rows128 {"passed": true, "mode": "add", "shape": [128, 128, 128], "tiles": 4, "mismatches": 0, "pl_time_ms": 0.38386383863838636}


add_b_only128 {"passed": true, "mode": "add", "shape": [128, 128, 128], "tiles": 4, "mismatches": 0, "pl_time_ms": 0.3443234432344323}


add_disjoint_cols128 {"passed": true, "mode": "add", "shape": [128, 128, 128], "tiles": 4, "mismatches": 0, "pl_time_ms": 0.4108041080410804}


add_M127_partial {"passed": true, "mode": "add", "shape": [127, 128, 128], "tiles": 4, "mismatches": 0, "pl_time_ms": 0.3934839348393484}


... (20 outputs left out)

sub_M70_partial {"passed": true, "mode": "sub", "shape": [70, 128, 128], "tiles": 4, "mismatches": 0, "pl_time_ms": 0.36336363363633634}


sub_same_cols128 {"passed": true, "mode": "sub", "shape": [128, 128, 128], "tiles": 4, "mismatches": 0, "pl_time_ms": 0.34948349483494834}


sub_small8_manual {"passed": true, "mode": "sub", "shape": [8, 8, 8], "tiles": 1, "mismatches": 0, "pl_time_ms": 0.08268082680826808}


ORIGINAL_SUITE_PASS: all 28 original cases match, SpMM/ADD/SUB


## 512 上限和数值边界验收

包含 512×512 乘法/加法/减法、K=512 满输入及舍入顺序检查，共 194 个硬件任务。此步骤可能需要几分钟，输入上传时间远大于计算时间是当前 MMIO 版本的正常现象。会生成 boundary_results 文件夹。


In [6]:
from nanshan import run_boundary_suite
boundary_reports = run_boundary_suite()

spmm_identity512 {"passed": true, "tiles": 64, "mismatches": 0, "pl_time_ms": 6.214722147221472}


add_identity512 {"passed": true, "tiles": 64, "mismatches": 0, "pl_time_ms": 5.271892718927189}


sub_cancel512 {"passed": true, "tiles": 64, "mismatches": 0, "pl_time_ms": 5.271892718927189}


dense_k512 {"passed": true, "tiles": 1, "mismatches": 0, "pl_time_ms": 5.83779837798378}


rounding_k512 {"passed": true, "tiles": 1, "mismatches": 0, "pl_time_ms": 0.909409094090941}


BOUNDARY_SUITE_PASS: 512-size SpMM/ADD/SUB and K=512 numerical boundaries


## 保存本次板测证据

三个成功标记分别为 THREE_MODE_SMOKE_PASS、ORIGINAL_SUITE_PASS、BOUNDARY_SUITE_PASS。按 Ctrl+S 保存 Notebook，再下载 board_environment.json、full_results 和 boundary_results 文件夹。

只有真实板卡上的原工程回归及上限验收都完成，才能将本版本标记为功能移植验收通过。硬件周期时间、上传/读回时间和软件端完整耗时需要分开解释，不能将硬件周期时间直接当作端到端加速比。
